In [1]:
"""
任务4：导出P4最终模型逐年指标和逐样本预测
基于XGBRFRegressor + Optuna MOBO，导出每个测试年份的预测值和指标。
"""
import pandas as pd
import numpy as np
import os
import warnings
import optuna
from optuna.samplers import TPESampler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRFRegressor
from sklearn.model_selection import LeaveOneGroupOut

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

DATA_PATH = r"D:\uv_py\xgb\data\P4_Cleaned_Dataset.csv"
OUTPUT_DIR = r"D:\uv_py\xgb\answer_todos\outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

OPTUNA_TRIALS = 30
RANDOM_SEED = 42
META_COLS = ["Year", "Zone", "latitude", "longitude", "yield"]

def calculate_metrics(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    mean_true = np.mean(y_true)
    rrmse = (rmse / mean_true) * 100 if mean_true != 0 else 0
    mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-8))) * 100
    den = np.sum((np.abs(y_pred - mean_true) + np.abs(y_true - mean_true)) ** 2)
    d_index = 1 - (np.sum((y_pred - y_true) ** 2) / den) if den != 0 else 0
    return {
        "R2": round(float(r2), 3), "RMSE": round(float(rmse), 3),
        "RRMSE(%)": round(float(rrmse), 3), "MAE": round(float(mae), 3),
        "MAPE(%)": round(float(mape), 3), "d-index": round(float(d_index), 3),
    }


In [2]:
def objective(trial, X_np, y_np, groups):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 50, 200, step=50),
        "max_depth": trial.suggest_int("max_depth", 6, 12),
        "colsample_bynode": trial.suggest_float("colsample_bynode", 0.3, 0.9),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "tree_method": "gpu_hist", "random_state": RANDOM_SEED, "n_jobs": -1,
    }
    logo = LeaveOneGroupOut()
    mae_scores = []
    for train_idx, val_idx in logo.split(X_np, y_np, groups):
        X_tr, X_val = X_np[train_idx], X_np[val_idx]
        y_tr, y_val = y_np[train_idx], y_np[val_idx]
        model = XGBRFRegressor(**params)
        model.fit(X_tr, y_tr)
        preds = model.predict(X_val)
        mae_scores.append(mean_absolute_error(y_val, preds))
    return np.mean(mae_scores)

RAW_METEO = ["Tmean", "PPT", "SM", "GDD", "FDD", "VPD", "VPD_max"]
RAW_RS = ["NDVI", "NDVI_max", "NDWI", "NIRv", "EVI", "EVI_max"]
SOIL = ["Sand", "Clay", "SOC"]
STAGES = ["P1", "P2", "P3", "P4"]

print(f"{'='*70}")
print(">>> P4 Final MOBO + Predict Export (raw features only)")
print(f"{'='*70}")

df = pd.read_csv(DATA_PATH)
all_cols = [c for c in df.columns if c not in META_COLS]

selected_features = list(SOIL)
for stage in STAGES:
    for var in RAW_METEO + RAW_RS:
        col = f"{stage}_{var}"
        if col in all_cols:
            selected_features.append(col)

print(f">>> Using {len(selected_features)} raw features (3 soil + 4 stages x 13 vars)")
print(f"    Optuna trials: {OPTUNA_TRIALS}, total fits: {6 * OPTUNA_TRIALS * 6}")

years = sorted(df["Year"].unique())
all_metrics = []
all_predictions = []
all_best_params = []

for test_year in years:
    train_df = df[df["Year"] != test_year]
    test_df = df[df["Year"] == test_year]
    X_train_np = train_df[selected_features].values
    y_train_np = train_df["yield"].values
    groups_train = train_df["Year"].values
    X_test_np = test_df[selected_features].values
    y_test_np = test_df["yield"].values

    print(f"[Year {test_year}] Optuna({len(selected_features)}feat)...", end="", flush=True)
    study = optuna.create_study(direction="minimize", sampler=TPESampler(seed=RANDOM_SEED))
    study.optimize(lambda trial: objective(trial, X_train_np, y_train_np, groups_train),
                   n_trials=OPTUNA_TRIALS, show_progress_bar=True)

    best_trial = study.best_trial
    best_params = {k: v for k, v in best_trial.params.items()}
    best_params.update({"tree_method": "gpu_hist", "random_state": RANDOM_SEED, "n_jobs": -1})

    final_model = XGBRFRegressor(**best_params)
    final_model.fit(X_train_np, y_train_np)
    y_pred = final_model.predict(X_test_np)

    m = calculate_metrics(y_test_np, y_pred)
    m["Test_Year"] = str(int(test_year))
    m["n_features"] = len(selected_features)
    all_metrics.append(m)

    all_predictions.append(pd.DataFrame({
        "Year": test_df["Year"].values,
        "Zone": test_df["Zone"].values,
        "latitude": test_df["latitude"].values,
        "longitude": test_df["longitude"].values,
        "yield_true": y_test_np,
        "yield_pred": y_pred,
        "error": y_test_np - y_pred,
    }))

    bp = {"Test_Year": str(int(test_year)), "n_features": len(selected_features)}
    bp.update({k: best_params.get(k) for k in ["n_estimators", "max_depth", "colsample_bynode", "subsample"]})
    all_best_params.append(bp)

    print(f"  Done. R2={m['R2']:.3f} MAPE={m['MAPE(%)']:.2f}%")

all_y_true = np.concatenate([p["yield_true"].values for p in all_predictions])
all_y_pred = np.concatenate([p["yield_pred"].values for p in all_predictions])
global_m = calculate_metrics(all_y_true, all_y_pred)
global_m["Test_Year"] = "Overall_Pooled"
global_m["n_features"] = len(selected_features)
all_metrics.append(global_m)

print(f"Overall: R2={global_m['R2']:.3f} MAPE={global_m['MAPE(%)']:.2f}%")

pd.DataFrame(all_metrics).to_csv(os.path.join(OUTPUT_DIR, "P4_Final_Model_ByYear.csv"), index=False)
pd.concat(all_predictions, ignore_index=True).to_csv(os.path.join(OUTPUT_DIR, "P4_Final_Model_Predictions.csv"), index=False)
pd.DataFrame(all_best_params).to_csv(os.path.join(OUTPUT_DIR, "P4_Final_Model_BestParams.csv"), index=False)

print("Done exporting.")


>>> P4 Final MOBO + Predict Export (raw features only)
>>> Using 55 raw features (3 soil + 4 stages x 13 vars)
    Optuna trials: 30, total fits: 1080
[Year 2016] Optuna(55feat)...

  0%|          | 0/30 [00:00<?, ?it/s]

  Done. R2=0.188 MAPE=8.89%
[Year 2017] Optuna(55feat)...

  0%|          | 0/30 [00:00<?, ?it/s]

  Done. R2=0.266 MAPE=10.90%
[Year 2018] Optuna(55feat)...

...

  Done. R2=0.332 MAPE=9.86%
[Year 2020] Optuna(55feat)...

  0%|          | 0/30 [00:00<?, ?it/s]

  Done. R2=0.279 MAPE=9.05%
[Year 2021] Optuna(55feat)...

  0%|          | 0/30 [00:00<?, ?it/s]

  Done. R2=0.385 MAPE=9.22%
Overall: R2=0.318 MAPE=9.57%
Done exporting.
